# 따릉이 스테이션 별 수요도 예측 Baseline

## 환경 설정

In [1]:
# ==========================================
# 통합 라이브러리 설정 (Master Setup)
# ==========================================
import os
import sys
import re
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from datetime import datetime, date, timedelta
from concurrent.futures import ThreadPoolExecutor
import geopandas as gpd
from shapely import wkt
from IPython.display import display, HTML

# ------------------------------------------
# 데이터베이스 및 환경 설정
# ------------------------------------------
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, Column, Integer, String, Float, DateTime, Date, Text, func
from sqlalchemy.orm import Mapped, mapped_column, Session

# ------------------------------------------
# Scikit-learn 모델 및 유틸리티
# ------------------------------------------
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ------------------------------------------
# 모델링 및 튜닝 도구
# ------------------------------------------
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import optuna

# ------------------------------------------
# 시각화 및 진행률 표시 도구
# ------------------------------------------
from tqdm.notebook import tqdm

# ==========================================
# 프로젝트 경로 설정 및 환경변수 로드
# ==========================================
# 상위 폴더(프로젝트 루트)를 모듈 검색 경로에 추가
sys.path.append(os.path.dirname(os.getcwd()))

# .env 파일 로드
load_dotenv()


# ==========================================
# 시각화 및 전역 환경 설정
# ==========================================
# 그래프에서 음수 부호(-) 깨짐 방지
plt.rcParams["axes.unicode_minus"] = False
# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 난수 시드 고정
SEED = 42
np.random.seed(SEED)

print("\n========== 데이터 분석 환경 설정 완료 ==========")
print(f"설정된 시드 값: {SEED}")
print("라이브러리 로드 완료")


========== 데이터 분석 환경 설정 완료 ==========
설정된 시드 값: 42
라이브러리 로드 완료


## 데이터베이스 연결

In [2]:
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from functools import reduce

# ==========================================
# 환경 설정 및 DB 연결
# ==========================================
load_dotenv()

DB_USER     = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD", "password")
DB_HOST     = os.getenv("DB_HOST", "localhost")
DB_PORT     = os.getenv("DB_PORT", "3306")
DB_NAME     = os.getenv("DB_NAME", "seoul_bike")

DATABASE_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

# 데이터베이스 연결 확인
with engine.connect() as conn:
    if conn.execute(text("SELECT 1")).scalar() == 1:
        print("\n========== 데이터베이스 연결 성공 ==========")
    else:
        print("\n========== 데이터베이스 연결 실패 ==========")


# ==========================================
# 데이터베이스 테이블 로드 (DataFrame)
# ==========================================
print("========== 각 테이블 데이터 로드 시작 ==========")

# 환경 데이터
hourly_air_2024_df    = pd.read_sql_table("hourly_air_2024", con=engine)
hourly_precip_2024_df = pd.read_sql_table("hourly_precip_2024", con=engine)
hourly_snow_2024_df   = pd.read_sql_table("hourly_snow_2024", con=engine)
hourly_temp_2024_df   = pd.read_sql_table("hourly_temp_2024", con=engine)
rt_air_df             = pd.read_sql_table("rt_air", con=engine)
rt_weather_df         = pd.read_sql_table("rt_weather", con=engine)

# 인프라 데이터
infra_business_df     = pd.read_sql_table("infra_business", con=engine)
infra_park_df         = pd.read_sql_table("infra_park", con=engine)
infra_river_df        = pd.read_sql_table("infra_river", con=engine)
infra_school_df       = pd.read_sql_table("infra_school", con=engine)
infra_univ_df         = pd.read_sql_table("infra_univ", con=engine)
infra_subway_df         = pd.read_sql_table("infra_subway", con=engine)


# 인구 데이터
pop_flow_2024_df      = pd.read_sql_table("pop_flow_2024", con=engine)
pop_living_2024_df    = pd.read_sql_table("pop_living_2024", con=engine)

# 따릉이 및 기타 데이터
korea_holidays_df     = pd.read_sql_table("korea_holidays", con=engine)
station_loc_df        = pd.read_sql_table("station_loc", con=engine)
rt_bike_status_df     = pd.read_sql_table("rt_bike_status", con=engine)

print("========== 데이터 로드 완료 ==========")


========== 데이터베이스 연결 성공 ==========
========== 각 테이블 데이터 로드 시작 ==========
========== 데이터 로드 완료 ==========


### 2024년 따릉이 대여 및 반납 이력

In [ ]:
# ==========================================
# 데이터베이스 연결 및 설정
# ==========================================
table_name = "rent_history_2024"
chunk_size = 100000  # 한 번에 10만 행씩 분할 로드

print(f"\n========== {table_name} 데이터 로드 시작 ==========")
start_time = time.time()

# chunksize를 설정하여 데이터를 분할 로드하는 이터레이터 생성
chunk_iterator = pd.read_sql_table(
    table_name,
    con=engine,
    chunksize=chunk_size
)

df_list = []
total_rows = 0

# 분할된 데이터를 하나씩 꺼내서 리스트에 병합
for chunk in chunk_iterator:
    df_list.append(chunk)
    total_rows += len(chunk)

# 리스트에 모인 조각들을 하나의 데이터프레임으로 병합
rent_history_2024_df = pd.concat(df_list, ignore_index=True)

end_time = time.time()

print("========== 데이터 로드 완료 ==========")


# ==========================================
# 로드 결과 및 메모리 사용량 확인
# ==========================================
print(f"총 로드된 행 수: {len(rent_history_2024_df):,}행")
print(f"데이터 로드 소요 시간: {end_time - start_time:.2f}초")

# 데이터프레임 메모리 사용량 계산 (MB 단위)
memory_mb = rent_history_2024_df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"데이터프레임 메모리 사용량: {memory_mb:.2f} MB")

## 전처리

### 따릉이 위치 데이터

In [3]:
# ==========================================
# 환경 변수 로드 및 설정
# ==========================================
KAKAO_REST_API_KEY = os.getenv("KAKAO_REST_API_KEY")

# API가 끝내 찾지 못한 대여소의 확실한 수동 좌표 사전
HARDCODED_COORDS = {
    'ST-1066': (37.55290, 126.83650), # 강서구 내발산동 (마곡수명산파크2단지 교차로)
    'ST-1068': (37.54897, 126.84852), # 강서구 화곡동 (한국여자 농구연맹 건너편)
    'ST-1073': (37.50325, 127.12782), # 송파구 오금동 (오금역 7번출구)
    'ST-1074': (37.49830, 127.13454), # 송파구 가락동 (개롱역 3번출구)
    'ST-1090': (37.48083, 127.12933), # 송파구 충민로 (송파 글마루도서관)
    'ST-1091': (37.50743, 127.10123), # 송파구 삼학사로 (석촌호수 교차로)
    'ST-1255': (37.56847, 126.84803), # 강서구 허준로 (기쁜우리복지관)
    'ST-1318': (37.53424, 126.89736), # 영등포구 선유로 (양평동6차현대아파트)
    'ST-1412': (37.50383, 127.13876), # 송파구 동남로24길 (우창아파트 상가)
    'ST-1415': (37.48161, 127.14361), # 송파구 위례광장로 (위례동 주민센터)
    'ST-2':    (37.55088, 126.91039), # 마포구 동교로8안길 23
    'ST-415':  (37.51980, 126.88937), # 영등포구 선유로 82 (KB국민은행 문래동지점 앞)
    'ST-423':  (37.52784, 126.92873), # 영등포구 여의나루로 96 (구 MBC)
    'ST-989':  (37.54955, 126.91071)  # 마포구 월드컵로5길 11 (합정동 주민센터)
}


# ==========================================
# 카카오 API 검색 함수 정의
# ==========================================
def preprocess_station_loc_1(address_1, address_2):
    """주소를 받아 카카오 API를 통해 위도(lat), 경도(lon)를 반환하는 함수"""
    addr1 = address_1 if pd.notna(address_1) else ""
    addr2 = address_2 if pd.notna(address_2) else ""

    query_full = f"{addr1} {addr2}".strip()
    query_partial = addr1.strip()

    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_REST_API_KEY}"}

    def preprocess_station_loc_2(query):
        if not query:
            return None
        try:
            response = requests.get(url, headers=headers, params={"query": query})
            response.raise_for_status()
            documents = response.json().get('documents')

            if documents:
                return float(documents[0]['y']), float(documents[0]['x'])

        except Exception as e:
            print(f"API 요청 실패 ({query}): {e}")
        return None

    # 전체 주소로 검색
    result = preprocess_station_loc_2(query_full)
    if result:
        return result

    # 전체 검색 실패 시, 부분 주소로 재검색
    if query_partial and query_partial != query_full:
        result = preprocess_station_loc_2(query_partial)
        if result:
            return result

    return 0.0, 0.0


# ==========================================
# 전처리 및 데이터 정제 메인 로직
# ==========================================
print("\n========== 위치 데이터(station_loc_df) 통합 전처리 시작 ==========")

# 기준키(대여소ID) 공백 제거 및 문자열 통일
station_loc_df['station_id'] = station_loc_df['station_id'].astype(str).str.strip()

# 위도나 경도가 0.0인 누락된 데이터 필터링
missing_mask = (station_loc_df['lat'] == 0.0) | (station_loc_df['lon'] == 0.0)

# 카카오 API 활용 순차 검색
for idx, row in station_loc_df[missing_mask].iterrows():
    new_lat, new_lon = preprocess_station_loc_1(row['address_1'], row['address_2'])
    if new_lat != 0.0 and new_lon != 0.0:
        station_loc_df.at[idx, 'lat'] = new_lat
        station_loc_df.at[idx, 'lon'] = new_lon

# API 검색 실패 건에 대한 수동 하드코딩 좌표 주입
for st_id, coords in HARDCODED_COORDS.items():
    lat, lon = coords
    idx = station_loc_df[station_loc_df['station_id'] == st_id].index

    if not idx.empty:
        station_loc_df.loc[idx, 'lat'] = lat
        station_loc_df.loc[idx, 'lon'] = lon


# ==========================================
# 최종 결과 검증 및 출력
# ==========================================
print("\n========== 전처리 완료 및 최종 점검 ==========")

# 데이터프레임 요약 정보 출력
print(station_loc_df.info())

# 최종 누락 건수 확인
failed_station_loc_df = station_loc_df[(station_loc_df['lat'] == 0.0) | (station_loc_df['lon'] == 0.0)]
print(f"\n최종 위경도 누락(0.0) 잔여 건수: {len(failed_station_loc_df)}건")

if not failed_station_loc_df.empty:
    print("처리되지 않은 데이터:")
    print(failed_station_loc_df[['station_id', 'address_1', 'address_2']])
else:
    print("========== 따릉이 위치 데이터 최종 전처리 완료 ==========")


========== 위치 데이터(station_loc_df) 통합 전처리 시작 ==========

========== 전처리 완료 및 최종 점검 ==========
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 846 entries, 0 to 845
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   station_id  846 non-null    object        
 1   address_1   846 non-null    object        
 2   address_2   846 non-null    object        
 3   lat         846 non-null    float64       
 4   lon         846 non-null    float64       
 5   district    846 non-null    object        
 6   grid_x      846 non-null    int64         
 7   grid_y      846 non-null    int64         
 8   created_at  846 non-null    datetime64[ns]
dtypes: datetime64[ns](1), float64(2), int64(2), object(4)
memory usage: 59.6+ KB
None

최종 위경도 누락(0.0) 잔여 건수: 0건
========== 따릉이 위치 데이터 최종 전처리 완료 ==========


### 환경 데이터

In [4]:
# ==========================================
# 환경 데이터 전처리 함수
# ==========================================
def preprocess_env(df, col_name):
    """
    환경 데이터의 이상치 제거, 날짜 변환 및 1시간 단위 리샘플링 수행
    """
    df = df.copy()

    # 불필요한 ID 컬럼 제거
    if 'id' in df.columns:
        df = df.drop(columns=['id'])

    # 날짜 변환 및 이상치(-9) 처리
    df['measure_date'] = pd.to_datetime(df['measure_date'])
    df.replace([-9, -9.0], np.nan, inplace=True)

    # 1시간 단위 리샘플링 후 평균값 계산
    df_resampled = (
        df.set_index('measure_date')
        .groupby('region_name')[col_name]
        .resample('1h')
        .mean()
        .reset_index()
    )

    return df_resampled


# ==========================================
# 데이터 전처리 실행
# ==========================================
print("\n========== 환경 데이터 전처리 시작 ==========")

processed_air    = preprocess_env(hourly_air_2024_df, 'pm10')
processed_temp   = preprocess_env(hourly_temp_2024_df, 'temperature')
processed_precip = preprocess_env(hourly_precip_2024_df, 'precipitation')
processed_snow   = preprocess_env(hourly_snow_2024_df, 'snowfall')

print("========== 환경 데이터 전처리 완료 ==========")


# ==========================================
# 최종 데이터 통합(Merge)
# ==========================================
print("========== 환경 데이터 통합 시작 ==========")

env_dfs = [processed_air, processed_temp, processed_precip, processed_snow]

# reduce를 활용한 외부 조인(outer join) 수행
env_master_2024_df = reduce(
    lambda left, right: pd.merge(left, right, on=['measure_date', 'region_name'], how='outer'),
    env_dfs
)

# 최종 결측치 처리
env_master_2024_df['precipitation'] = env_master_2024_df['precipitation'].fillna(0)
env_master_2024_df['snowfall'] = env_master_2024_df['snowfall'].fillna(0)

print("========== 환경 데이터 통합 종료 ==========")


========== 환경 데이터 전처리 시작 ==========
========== 환경 데이터 전처리 완료 ==========
========== 환경 데이터 통합 시작 ==========
========== 환경 데이터 통합 종료 ==========


In [5]:
# ==========================================
# DB(SQL) 저장 로직
# ==========================================
table_name = 'env_master_2024'
env_master_2024_df.to_sql(
    name=table_name,
    con=engine,
    if_exists='replace',
    index=False
)
print(f"========== DB 적재 성공 ==========")

========== DB 적재 성공 ==========


### 인구 데이터

In [6]:
# ==========================================
# 생활인구 데이터 전처리
# ==========================================
print("\n========== 생활인구 데이터 전처리 시작 ==========")
district_map = {'11500': '강서구', '11560': '영등포구', '11440': '마포구', '11710': '송파구'}
pop_living_2024_df['district_name'] = pop_living_2024_df['adstrd_code_se'].map(district_map)

# 컬럼명 정리
pop_living_2024_df.rename(columns={'tot_lvpop_co': 'lvgpop_tot'}, inplace=True)

# 연령대별 합산
pop_living_2024_df['lvgpop_10s'] = pop_living_2024_df[['male_f10t14_lvpop_co', 'male_f15t19_lvpop_co', 'female_f10t14_lvpop_co', 'female_f15t19_lvpop_co']].sum(axis=1)
pop_living_2024_df['lvgpop_20s'] = pop_living_2024_df[['male_f20t24_lvpop_co', 'male_f25t29_lvpop_co', 'female_f20t24_lvpop_co', 'female_f25t29_lvpop_co']].sum(axis=1)
pop_living_2024_df['lvgpop_30s'] = pop_living_2024_df[['male_f30t34_lvpop_co', 'male_f35t39_lvpop_co', 'female_f30t34_lvpop_co', 'female_f35t39_lvpop_co']].sum(axis=1)
pop_living_2024_df['lvgpop_40s'] = pop_living_2024_df[['male_f40t44_lvpop_co', 'male_f45t49_lvpop_co', 'female_f40t44_lvpop_co', 'female_f45t49_lvpop_co']].sum(axis=1)
pop_living_2024_df['lvgpop_50s'] = pop_living_2024_df[['male_f50t54_lvpop_co', 'male_f55t59_lvpop_co', 'female_f50t54_lvpop_co', 'female_f55t59_lvpop_co']].sum(axis=1)
pop_living_2024_df['lvgpop_60up'] = pop_living_2024_df[['male_f60t64_lvpop_co', 'male_f65t69_lvpop_co', 'male_f70t74_lvpop_co', 'female_f60t64_lvpop_co', 'female_f65t69_lvpop_co', 'female_f70t74_lvpop_co']].sum(axis=1)

# 병합을 위한 키(Key) 생성
pop_living_2024_df = pop_living_2024_df[['stdr_de_id', 'tmzon_pd_se', 'district_name', 'lvgpop_tot', 'lvgpop_10s', 'lvgpop_20s', 'lvgpop_30s', 'lvgpop_40s', 'lvgpop_50s', 'lvgpop_60up']].copy()
pop_living_2024_df['date_str'] = pop_living_2024_df['stdr_de_id'].astype(str)
pop_living_2024_df['hour_str'] = pop_living_2024_df['tmzon_pd_se'].astype(str).str.zfill(2)


# ==========================================
# 2024년 인구 통합 마스터 생성
# ==========================================
print("========== 인구 통합 마스터 생성 시작 ==========")
date_rng = pd.date_range(start='2024-01-01 00:00:00', end='2024-12-31 23:00:00', freq='h')
districts = ['강서구', '영등포구', '마포구', '송파구']

pop_master_2024_list = []
for dist in districts:
    df_temp = pd.DataFrame(date_rng, columns=['datetime'])
    df_temp['district_name'] = dist
    df_temp['date_str'] = df_temp['datetime'].dt.strftime('%Y%m%d')
    df_temp['hour_str'] = df_temp['datetime'].dt.strftime('%H')
    df_temp['weekday'] = df_temp['datetime'].dt.weekday
    df_temp['quarter_str'] = '2024' + df_temp['datetime'].dt.quarter.astype(str)
    pop_master_2024_list.append(df_temp)

pop_master_2024_df = pd.concat(pop_master_2024_list, ignore_index=True)


# ==========================================
# 유동인구 분배 및 병합 로직
# ==========================================
print("========== 유동인구 데이터 병합 및 계산 시작 ==========")
pop_flow_2024_df = pd.merge(
    pop_master_2024_df,
    pop_flow_2024_df,
    left_on=['quarter_str', 'district_name'],
    right_on=['stdr_yyqu_cd', 'signgu_cd_nm'],
    how='left'
)

# 분배 기준값 설정
time_divisors = {
    '00': 6, '01': 6, '02': 6, '03': 6, '04': 6, '05': 6,
    '06': 5, '07': 5, '08': 5, '09': 5, '10': 5,
    '11': 3, '12': 3, '13': 3, '14': 3, '15': 3, '16': 3,
    '17': 4, '18': 4, '19': 4, '20': 4,
    '21': 3, '22': 3, '23': 3
}
weekday_col_map = {
    0: 'mon_flpop_co', 1: 'tues_flpop_co', 2: 'wed_flpop_co',
    3: 'thur_flpop_co', 4: 'fri_flpop_co', 5: 'sat_flpop_co', 6: 'sun_flpop_co'
}

def calc_flow_pop(row):
    """시간대별 유동인구 분배 계산 함수"""
    h, wd = row['hour_str'], row['weekday']
    div = time_divisors.get(h, 1)

    # 시간대별 카테고리 매핑
    if h in ['00','01','02','03','04','05']: time_pop = row['tmzon_00_06_flpop_co']
    elif h in ['06','07','08','09','10']: time_pop = row['tmzon_06_11_flpop_co']
    elif h in ['11','12','13']: time_pop = row['tmzon_11_14_flpop_co']
    elif h in ['14','15','16']: time_pop = row['tmzon_14_17_flpop_co']
    elif h in ['17','18','19','20']: time_pop = row['tmzon_17_21_flpop_co']
    else: time_pop = row['tmzon_21_24_flpop_co']

    day_pop = row[weekday_col_map[wd]]
    tot_pop = row['tot_flpop_co']

    if pd.notna(tot_pop) and tot_pop > 0:
        est_tot_flwpop = (time_pop / div) * (day_pop / tot_pop) / 13
        return pd.Series([
            est_tot_flwpop,
            est_tot_flwpop * (row['agrde_10_flpop_co'] / tot_pop),
            est_tot_flwpop * (row['agrde_20_flpop_co'] / tot_pop),
            est_tot_flwpop * (row['agrde_30_flpop_co'] / tot_pop),
            est_tot_flwpop * (row['agrde_40_flpop_co'] / tot_pop),
            est_tot_flwpop * (row['agrde_50_flpop_co'] / tot_pop),
            est_tot_flwpop * (row['agrde_60_above_flpop_co'] / tot_pop)
        ])
    return pd.Series([0, 0, 0, 0, 0, 0, 0])

# 유동인구 계산 적용
pop_flow_2024_df[['flwpop_tot', 'flwpop_10s', 'flwpop_20s', 'flwpop_30s', 'flwpop_40s', 'flwpop_50s', 'flwpop_60up']] = \
    pop_flow_2024_df.apply(calc_flow_pop, axis=1)


# ==========================================
# 최종 병합 및 정리
# ==========================================
print("========== 최종 데이터 병합 완료 ==========")
pop_master_2024_df = pd.merge(
    pop_flow_2024_df,
    pop_living_2024_df,
    on=['date_str', 'hour_str', 'district_name'],
    how='left'
)

# 핵심 컬럼만 추출
pop_master_2024_cols = [
    'datetime', 'district_name',
    'flwpop_tot', 'flwpop_10s', 'flwpop_20s', 'flwpop_30s', 'flwpop_40s', 'flwpop_50s', 'flwpop_60up',
    'lvgpop_tot', 'lvgpop_10s', 'lvgpop_20s', 'lvgpop_30s', 'lvgpop_40s', 'lvgpop_50s', 'lvgpop_60up'
]
pop_master_2024_df = pop_master_2024_df[pop_master_2024_cols]


========== 생활인구 데이터 전처리 시작 ==========
========== 인구 통합 마스터 생성 시작 ==========
========== 유동인구 데이터 병합 및 계산 시작 ==========
========== 최종 데이터 병합 완료 ==========


In [7]:
# ==========================================
# DB(SQL) 저장 로직
# ==========================================
table_name = 'pop_master_2024'
pop_master_2024_df.to_sql(
    name=table_name,
    con=engine,
    if_exists='replace',
    index=False
)
print(f"========== DB 적재 성공 ==========")

========== DB 적재 성공 ==========


### 인프라 데이터

In [8]:
# ==========================================
# 공간 데이터 전처리 및 피처 연산 함수
# ==========================================
def preprocess_gdf(df, wkt_col=None):
    """
    자동 컬럼 감지를 통한 GeoDataFrame 변환 전처리 수행
    """
    if df.empty:
        return gpd.GeoDataFrame()

    # 하천 데이터 처리
    if wkt_col:
        df = df.dropna(subset=[wkt_col]).copy()
        df['geometry'] = df[wkt_col].apply(
            lambda x: wkt.loads(str(x)) if pd.notna(x) and str(x) != 'None' else None
        )
        df = df.dropna(subset=['geometry'])
        gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

    # 점 데이터 처리
    else:
        lat_col = 'latitude' if 'latitude' in df.columns else 'lat' if 'lat' in df.columns else None
        lon_col = 'longitude' if 'longitude' in df.columns else 'lon' if 'lon' in df.columns else 'lot' if 'lot' in df.columns else None

        # 위경도 컬럼이 없으면 빈 데이터 반환
        if not lat_col or not lon_col:
            return gpd.GeoDataFrame()

        df = df.dropna(subset=[lat_col, lon_col]).copy()
        gdf = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df[lon_col].astype(float), df[lat_col].astype(float)),
            crs="EPSG:4326"
        )

    # 좌표계 변환 및 빈 공간 데이터 제거
    gdf = gdf.to_crs(epsg=5179)
    gdf = gdf[~gdf.is_empty]

    return gdf


def count_infra(target_gdf, source_gdf, radius_m, col_name):
    """
    지정된 반경 내 대상 인프라 개수 계산
    """
    if source_gdf is None or source_gdf.empty:
        return pd.Series(0, index=target_gdf.index, name=col_name)

    buffer_gdf = target_gdf.copy()
    buffer_gdf['geometry'] = buffer_gdf.geometry.buffer(radius_m)
    joined = gpd.sjoin(buffer_gdf, source_gdf, how='left', predicate='intersects')
    count = joined.groupby(joined.index)['index_right'].count()

    return count.rename(col_name)


def cal_nearest_infra(target_gdf, source_gdf, col_name):
    """
    대상 인프라까지의 최단 거리 계산
    """
    if source_gdf is None or source_gdf.empty:
        return pd.DataFrame({col_name: [np.nan]*len(target_gdf)}, index=target_gdf.index)

    nearest = gpd.sjoin_nearest(target_gdf, source_gdf, distance_col=col_name)
    nearest = nearest[~nearest.index.duplicated(keep='first')]

    return nearest[[col_name]]


# ==========================================
# 공간 데이터 변환 실행
# ==========================================
print("\n========== 공간 데이터 변환 시작 ==========")

# 대여소 원본 데이터 전처리
station_loc_df['lat'] = pd.to_numeric(station_loc_df['lat'], errors='coerce')
station_loc_df['lon'] = pd.to_numeric(station_loc_df['lon'], errors='coerce')
station_loc_gdf = preprocess_gdf(station_loc_df)

# 공원 데이터 면적 추출 및 좌표 매핑
infra_park_df['park_area'] = infra_park_df['area'].apply(lambda x: float(re.sub(r'[^0-9.]', '', str(x))) if pd.notna(x) else np.nan)
infra_park_df['lon'] = infra_park_df['xcrd_g'].fillna(infra_park_df['xcrd'])
infra_park_df['lat'] = infra_park_df['ycrd_g'].fillna(infra_park_df['ycrd'])

# 지하철 데이터 좌표 변환
if 'lat' in infra_subway_df.columns:
    infra_subway_df['lat'] = pd.to_numeric(infra_subway_df['lat'], errors='coerce')
if 'lot' in infra_subway_df.columns:
    infra_subway_df['lot'] = pd.to_numeric(infra_subway_df['lot'], errors='coerce')
elif 'lon' in infra_subway_df.columns:
    infra_subway_df['lon'] = pd.to_numeric(infra_subway_df['lon'], errors='coerce')

# 일괄 변환 생성
infra_park_gdf     = preprocess_gdf(infra_park_df)
infra_river_gdf    = preprocess_gdf(infra_river_df, wkt_col='geom_wkt')
infra_subway_gdf   = preprocess_gdf(infra_subway_df)
infra_business_gdf = preprocess_gdf(infra_business_df)
infra_univ_gdf     = preprocess_gdf(infra_univ_df)
infra_school_gdf   = preprocess_gdf(infra_school_df)

# 반경 카운트를 위한 교육 시설 통합
infra_edu_gdf = pd.concat([infra_school_gdf, infra_univ_gdf], ignore_index=True)

print("========== 공간 데이터 변환 완료 ==========")


# ==========================================
# 인프라 데이터 통합 연산
# ==========================================
print("\n========== 인프라 데이터 통합 시작 ==========")

infra_master_df = station_loc_gdf.copy()

# 반경 내 개수 계산
infra_master_df['subway_cnt_300m'] = count_infra(infra_master_df, infra_subway_gdf, 300, 'subway_cnt_300m')
infra_master_df['biz_cnt_300m']    = count_infra(infra_master_df, infra_business_gdf, 300, 'biz_cnt_300m')
infra_master_df['edu_cnt_500m']    = count_infra(infra_master_df, infra_edu_gdf, 500, 'edu_cnt_500m')
infra_master_df['park_cnt_500m']   = count_infra(infra_master_df, infra_park_gdf, 500, 'park_cnt_500m')
infra_master_df['river_cnt_1km']   = count_infra(infra_master_df, infra_river_gdf, 1000, 'river_cnt_1km')

# 최단 거리 계산
infra_master_df = infra_master_df.join(cal_nearest_infra(infra_master_df, infra_subway_gdf, 'dist_subway'))
infra_master_df = infra_master_df.join(cal_nearest_infra(infra_master_df, infra_river_gdf, 'dist_river'))

# 데이터프레임 컬럼 재배치
infra_master_df = infra_master_df.drop(columns=['geometry'])
base_cols = ['station_id', 'address_1', 'lat', 'lon', 'district']

actual_base_cols = [c for c in base_cols if c in infra_master_df.columns]
feature_cols = ['subway_cnt_300m', 'biz_cnt_300m', 'edu_cnt_500m', 'park_cnt_500m', 'river_cnt_1km', 'dist_subway', 'dist_river']
other_cols = [c for c in infra_master_df.columns if c not in actual_base_cols + feature_cols]

infra_master_df = infra_master_df[actual_base_cols + feature_cols + other_cols]

print("========== 인프라 데이터 통합 종료 ==========")


========== 공간 데이터 변환 시작 ==========
========== 공간 데이터 변환 완료 ==========

========== 인프라 데이터 통합 시작 ==========
========== 인프라 데이터 통합 종료 ==========


In [10]:
# ==========================================
# DB(SQL) 저장 로직
# ==========================================
table_name = 'infra_master'
infra_master_df.to_sql(
    name=table_name,
    con=engine,
    if_exists='replace',
    index=False
)
print(f"========== DB 적재 성공 ==========")

========== DB 적재 성공 ==========


### 수요 예측용 데이터

In [ ]:
# ==========================================
# 베이스 테이블 준비
# ==========================================
print("\n========== 수요 예측용 마스터 데이터 병합 시작 ==========")

# 원본 데이터 보존을 위해 copy() 사용
demand_predict_master_2024_df = rent_history_2024_df.copy()


# ==========================================
# 공간 및 위치 특성 결합
# ==========================================
demand_predict_master_2024_df = pd.merge(
    demand_predict_master_2024_df,
    station_loc_df[['station_id', 'district', 'lat', 'lon']],
    on='station_id',
    how='left'
)


# ==========================================
# 정적 인프라 특성 결합
# ==========================================
demand_predict_master_2024_df = pd.merge(
    demand_predict_master_2024_df,
    infra_master_df,
    on='station_id',
    how='left'
)


# ==========================================
# 동적 기상 특성 결합
# ==========================================
demand_predict_master_2024_df = pd.merge(
    demand_predict_master_2024_df,
    env_master_2024_df[['measure_date', 'region_name', 'temperature', 'precipitation', 'snowfall', 'pm10']],
    left_on=['datetime_hr', 'district'],
    right_on=['measure_date', 'region_name'],
    how='left'
)

# 조인 후 중복되는 우측 키 컬럼 삭제
demand_predict_master_2024_df.drop(columns=['measure_date', 'region_name'], inplace=True)


# ==========================================
# 동적 인구 특성 결합
# ==========================================
demand_predict_master_2024_df = pd.merge(
    demand_predict_master_2024_df,
    pop_master_2024_df,
    left_on=['datetime_hr', 'district'],
    right_on=['datetime', 'district_name'],
    how='left'
)

# 조인 후 중복되는 우측 키 컬럼 삭제
demand_predict_master_2024_df.drop(columns=['datetime', 'district_name'], inplace=True)


# ==========================================
# 시간 및 파생 특성 결합
# ==========================================
# datetime_hr에서 날짜(Date) 부분만 추출하여 임시 컬럼 생성
demand_predict_master_2024_df['temp_date'] = demand_predict_master_2024_df['datetime_hr'].dt.date
korea_holidays_df['holiday_date'] = pd.to_datetime(korea_holidays_df['holiday_date']).dt.date

demand_predict_master_2024_df = pd.merge(
    demand_predict_master_2024_df,
    korea_holidays_df[['holiday_date', 'holiday_name']],
    left_on='temp_date',
    right_on='holiday_date',
    how='left'
)

# 공휴일 여부(is_holiday) Boolean 컬럼 생성
demand_predict_master_2024_df['is_holiday'] = demand_predict_master_2024_df['holiday_name'].notna()

# 불필요한 임시 날짜 컬럼 및 명칭 컬럼 삭제
demand_predict_master_2024_df.drop(columns=['temp_date', 'holiday_date', 'holiday_name'], inplace=True)


# ==========================================
# 요일 및 주말 파생 변수 생성
# ==========================================
demand_predict_master_2024_df['day_of_week'] = demand_predict_master_2024_df['datetime_hr'].dt.dayofweek
demand_predict_master_2024_df['is_weekend'] = demand_predict_master_2024_df['day_of_week'] >= 5


# ==========================================
# 최종 결과물 할당 및 결측치 기본 처리
# ==========================================
# 수치형 데이터 결측치를 0으로 채우기
fill_zero_cols = [
    'precipitation', 'snowfall', 'subway_cnt_300m',
    'biz_cnt_300m', 'edu_cnt_500m', 'park_cnt_500m', 'river_cnt_1km'
]

for col in fill_zero_cols:
    if col in demand_predict_master_2024_df.columns:
        demand_predict_master_2024_df[col] = demand_predict_master_2024_df[col].fillna(0)


print("========== 수요 예측용 마스터 데이터 병합 종료 ==========")

# 최종 데이터프레임 정보 확인
demand_predict_master_2024_df.info()

In [ ]:
# ==========================================
# DB(SQL) 저장 로직
# ==========================================
table_name = 'demand_predict_master_2024'
demand_predict_master_2024_df.to_sql(
    name=table_name,
    con=engine,
    if_exists='replace',
    index=False
)
print(f"========== DB 적재 성공 ==========")

## 피처(X) / 타깃(Y) 분리

## Train / Validation / Test 3분할

## 평가 지표 함수 및 기본 모델 학습

# 따릉이 수요 예측(각자 진행)

## 여러 모델 비교(Ridge, RandomForest, XGBoost, LightGBM)

In [ ]:
# 수요용 예측 그거 불러와야함

## 앙상블(Voting Regressor)

# 최종

## 최종 모델 선택

## Test셋 최종 평가

## 모델 저장(pkl) 및 저장된 모델 검증